In [1]:
!pip3 install pandas


[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


# Recent bench reports — historical comparison

A focused view of how the per-algorithm wall-clock has evolved across the
last few `./build.sh bench` runs. The notebook is intentionally narrow:
it answers _"did the last commit regress or improve performance?"_ for
the length buckets that matter most (LONG, XL, XXL) and the four
algorithms we care about most.

Sister notebook: `bioseq/RESEARCH/BENCHMARKING_SPEEDUP/view_report.ipynb`
renders one full report in detail. This one trades depth for breadth so
regressions across the last few runs are obvious at a glance.

## What this notebook does

1. Walks `main/reporting/` and picks the **most recent up to 5** reports,
   ranked by the `<N>` suffix of `report_build_<N>.json` (higher `N` =
   newer; the bench writer auto-increments this counter).
2. Loads each report's `seq_length_vs_speed_mean` chart and computes the
   mean wall-clock per pair for every (algorithm, length-bucket) cell.
3. Renders one table per length bucket (LONG, XL, XXL) where rows are
   the four headline algorithms and columns are the selected reports in
   chronological order (oldest on the left, newest on the right).
4. The column header has two levels: the upper level groups consecutive
   reports that share a commit hash (so a re-run on the same commit
   reads as one block), and the lower level shows the report number.
   Full commit hashes are listed at the bottom — the header itself
   stays compact.
5. Cells are tinted **green** when the value is more than `tolerance_pct`
   _faster_ than the cell to its left, **red** when it is more than
   `tolerance_pct` _slower_. The first column has no neighbour to
   compare to and is left uncoloured. `tolerance_pct` is loaded from
   `regression_thresholds.json` so this view stays in sync with the
   regression check that the bench harness already enforces.

## Why this view exists

After landing a perf-sensitive change you want a one-screen answer to
_"did anything regress?"_ before opening the much larger per-report
notebook. Spotting a sudden red column in the LONG/XL/XXL tables is a
lot faster than staring at 30+ Plotly charts.

In [2]:
# ----- configuration + threshold load -----
# REPORTING_DIR is where ./build.sh bench writes report_build_<N>.json.
# THRESHOLDS_PATH is the same JSON the regression checker reads, so
# tolerance_pct used to colour cells matches what the bench enforces.

import json
import pathlib
import re

import pandas as pd
from IPython.display import display, Markdown

REPORTING_DIR   = pathlib.Path('../reporting/bioseq_bench')
THRESHOLDS_PATH = pathlib.Path('regression_thresholds.json')
MAX_REPORTS     = 10

with open(THRESHOLDS_PATH, 'r') as fh:
    thresholds_doc = json.load(fh)
TOLERANCE_PCT = float(thresholds_doc.get('tolerance_pct', 2.0))
print(f'tolerance_pct (from regression_thresholds.json): {TOLERANCE_PCT}%')

tolerance_pct (from regression_thresholds.json): 2.0%


In [3]:
# Discover report_build_<N>.json files. Pick the 5 with the highest
# <N> (newest by index), then re-sort ascending so the leftmost column
# in every table is the *oldest* of the selected reports — reading
# left-to-right then matches the order the runs happened.

report_pattern = re.compile(r'^report_build_(\d+)\.json$')
candidates = []
for path in REPORTING_DIR.glob('report_build_*.json'):
    m = report_pattern.match(path.name)
    if m:
        candidates.append((int(m.group(1)), path))
if not candidates:
    raise SystemExit(f'No report_build_*.json files found in {REPORTING_DIR.resolve()}')
candidates.sort(key=lambda t: t[0], reverse=True)
candidates = candidates[:MAX_REPORTS]
candidates.sort(key=lambda t: t[0])

reports = []
for idx, path in candidates:
    with open(path, 'r') as fh:
        data = json.load(fh)
    reports.append({
        'index':  idx,
        'path':   path,
        'commit': data.get('commit_hash', 'unknown'),
        'data':   data,
    })

# Assign one stable label per unique commit hash, ordered by first
# appearance in the chronological list. Reports that share a hash
# share a label; pandas MultiIndex will visually merge adjacent
# identical labels in the column header.
commit_order = []
commit_to_label = {}
for r in reports:
    c = r['commit']
    if c not in commit_to_label:
        commit_order.append(c)
        commit_to_label[c] = f'commit {len(commit_order)}'
    r['commit_label'] = commit_to_label[c]

print(f'Loaded {len(reports)} report(s):')
for r in reports:
    short = (r['commit'][:12] + '\u2026') if len(r['commit']) > 12 else r['commit']
    print(f'  report {r["index"]:>3}  {r["commit_label"]:<10}  {short:<14}  ({r["path"].name})')

Loaded 10 report(s):
  report  15  commit 1    35b9e5bea6d4…   (report_build_15.json)
  report  16  commit 1    35b9e5bea6d4…   (report_build_16.json)
  report  17  commit 1    35b9e5bea6d4…   (report_build_17.json)
  report  18  commit 2    bfda2ebce1a2…   (report_build_18.json)
  report  19  commit 2    bfda2ebce1a2…   (report_build_19.json)
  report  20  commit 2    bfda2ebce1a2…   (report_build_20.json)
  report  21  commit 3    09a230c403fd…   (report_build_21.json)
  report  22  commit 3    09a230c403fd…   (report_build_22.json)
  report  23  commit 3    09a230c403fd…   (report_build_23.json)
  report  24  commit 3    09a230c403fd…   (report_build_24.json)


In [4]:
# ----- algorithm + bucket definitions + helpers -----
# (canonical name in report.json, display label in the table row).
# The canonical names match the labels matrix_runner.cppm uses when it
# emits seq_length_vs_speed_mean traces.

ALGORITHMS = [
    ('minimap2 (DP chaining)',
     'minimap2 (DP, non-RMQ)'),
    ('bioseq KSW2 + scalar-DP + scalar-mm',
     'KSW2 + bioseq scalar + scalar minimizers'),
    ('bioseq KSW2 + SIMD-DP + two-stack-nthash-mm',
     'KSW2 + bioseq SIMD-DP + two-stack-nthash'),
    ('bioseq Block + SIMD-DP + two-stack-nthash-mm',
     'Block aligner + bioseq SIMD-DP + two-stack-nthash'),
]

# Length buckets we care about for the historical view. The boundaries
# match the bench harness's bucket definitions in matrix_runner.cppm.
LONG_BUCKET = (10_000,     1_000_000)
XL_BUCKET   = (5_000_000,  15_000_000)
XXL_BUCKET  = (50_000_000, 150_000_000)


def bucket_mean(report_data, algorithm_name, lo, hi):
    '''Mean wall-clock (ms) across all (query, target) pairs whose
    query length falls in [lo, hi). Returns None if the algorithm
    has no point in the bucket — typical when XL/XXL was not part
    of that run, or the algorithm is not present in the report.'''
    chart = report_data.get('charts', {}).get('seq_length_vs_speed_mean')
    if not chart:
        return None
    for trace in chart.get('plotly', {}).get('data', []):
        if trace.get('name') != algorithm_name:
            continue
        xs = trace.get('x', [])
        ys = trace.get('y', [])
        in_bucket = [y for x, y in zip(xs, ys)
                     if y is not None and lo <= x < hi]
        if not in_bucket:
            return None
        return sum(in_bucket) / len(in_bucket)
    return None


def build_bucket_dataframe(bucket_lo, bucket_hi):
    '''DataFrame: rows = ALGORITHMS, columns = MultiIndex(commit, report).'''
    columns = pd.MultiIndex.from_tuples(
        [(r['commit_label'], f'report {r["index"]}') for r in reports],
        names=['Commit', 'Report'],
    )
    rows = []
    index = []
    for canonical, display_name in ALGORITHMS:
        index.append(display_name)
        rows.append([bucket_mean(r['data'], canonical, bucket_lo, bucket_hi)
                     for r in reports])
    return pd.DataFrame(rows, index=index, columns=columns)


def colour_history(df, tolerance_pct):
    '''Tint each cell relative to the previous (left) cell in the same row.

    delta_pct = (current - prev) / prev * 100. Negative is faster
    (green), positive is slower (red). Cells inside the tolerance
    window stay uncoloured. The first column, and any cell whose
    left-hand neighbour is missing, are left uncoloured.

    Colour intensity scales with how far past tolerance the delta is,
    capped at 20 percentage points beyond tolerance for readability.
    '''
    def row_styles(values):
        styles = ['' for _ in values]
        prev = None
        for i, value in enumerate(values):
            if pd.isna(value):
                continue
            if prev is not None and prev > 0:
                delta_pct = (value - prev) / prev * 100.0
                if delta_pct < -tolerance_pct:
                    intensity = min(1.0, (abs(delta_pct) - tolerance_pct) / 20.0)
                    rb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb({rb}, 255, {rb}); '
                                 f'color: black; font-weight: 600;')
                elif delta_pct > tolerance_pct:
                    intensity = min(1.0, (delta_pct - tolerance_pct) / 20.0)
                    gb = int(220 - 90 * intensity)
                    styles[i] = (f'background-color: rgb(255, {gb}, {gb}); '
                                 f'color: black; font-weight: 600;')
            prev = value
        return styles
    return (df.style
              .apply(row_styles, axis=1)
              .format('{:.2f} ms', na_rep='\u2014')
              .set_table_styles([
                  {'selector': 'th',
                   'props': 'text-align: center; padding: 4px 10px;'},
                  {'selector': 'td',
                   'props': 'text-align: right; padding: 4px 10px; '
                            'font-variant-numeric: tabular-nums;'},
              ]))

## LONG bucket (10 kbp – 1 Mbp)

The most balanced regression signal: long enough that base-level
alignment dominates the wall-clock, short enough that every report
still has a healthy number of samples per (algorithm, error) cell, so
the means here are reasonably stable and a colour change usually
reflects a real speedup or regression rather than noise.

In [5]:
display(colour_history(build_bucket_dataframe(*LONG_BUCKET), TOLERANCE_PCT))

## XL bucket (5 Mbp – 15 Mbp)

Two samples per pair by default (one with `--fast`). Each pair takes
seconds of wall-clock, so a regression in this bucket is the single
biggest cost contributor to a typical full bench run. Colour changes
here matter even when the millisecond delta looks small in absolute
terms — the percentage swing is what the regression checker scores.

In [6]:
display(colour_history(build_bucket_dataframe(*XL_BUCKET), TOLERANCE_PCT))

## XXL bucket (50 Mbp – 150 Mbp)

Single-sample bucket — each cell is one measurement, so the noise
floor is high. Treat colouring here as a hint rather than a verdict:
a cell that flips between red and green across consecutive reports
is normal noise, while a genuine regression shows up as a sustained
red trend across two or more reports in a row.

In [7]:
display(colour_history(build_bucket_dataframe(*XXL_BUCKET), TOLERANCE_PCT))

## Commit hashes

Full commit hash for each `commit N` label used in the column headers
above. Use these to cross-reference `git show <hash>` (or open the
commit on GitHub) when a coloured cell points at a specific change.

In [8]:
lines = ['**Commit hashes:**', '']
for commit in commit_order:
    label = commit_to_label[commit]
    lines.append(f'* **{label}**: `{commit}`')
display(Markdown('\n'.join(lines)))

**Commit hashes:**

* **commit 1**: `35b9e5bea6d45d0c5cac7cbd647fde8adcc26dbc`
* **commit 2**: `bfda2ebce1a2aea72d52089b4929209a074eee50`
* **commit 3**: `09a230c403fd12cc5949ee55606d77a0fac9f46f`